# MOMENT imputation — DIMER live tutorial

This notebook uses the pretrained MOMENT reconstruction path for **patch-granular artificial masking**. It withholds an 8-step patch from a clean deterministic series, scores only deliberately hidden source-observed points, preserves all other observed values in the exported imputed series, and writes provenance.

The model works in 8-step patches. A caller mask is therefore patch-quantized before MOMENT sees it; the tutorial reports both source missingness and effective model masking.


## 1. Bootstrap the repository and locked runtime


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/moment-pipeline.git"
REPO_NAME = "moment-pipeline"

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "--no-deps", "-e", "."], check=True)
else:
    print(f"Repository checkout detected: {ROOT}")


## 2. Load the deterministic clean sample or BYOD


In [ ]:
import hashlib
import io
import json

import pandas as pd

USE_BYOD = False
if USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("BYOD upload is available when this notebook runs in Colab.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV.")
    name, payload = next(iter(uploaded.items()))
    frame = pd.read_csv(io.BytesIO(payload))
    print(f"Loaded BYOD: {name}")
else:
    sample_root = ROOT / "examples" / "sample-data"
    subprocess.run([sys.executable, str(sample_root / "generate_samples.py")], check=True)
    sample_path = sample_root / "moment_clean.csv"
    manifest = {}
    for line in (sample_root / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        digest, filename = line.split("  ", 1)
        manifest[filename] = digest
    observed = hashlib.sha256(sample_path.read_bytes()).hexdigest()
    assert observed == manifest[sample_path.name]
    frame = pd.read_csv(sample_path)
    print(f"Loaded verified synthetic sample: {sample_path}")
frame["timestamp"] = pd.to_datetime(frame["timestamp"])
print(frame.head())
print(f"rows={len(frame)}, channels={frame['channel'].nunique()}")


## 3. Canonicalize and create a patch-level holdout


In [ ]:
import numpy as np

from moment_pipeline import build_provenance, impute, load_moment, masked_point_metrics, to_windows

windows = to_windows(frame)
visible = np.ones_like(windows.input_mask, dtype=np.float32)
valid_positions = np.flatnonzero(windows.input_mask[0] == 1)
start = int(valid_positions[-64])
start -= start % windows.patch_length
visible[:, start : start + windows.patch_length] = 0.0
print("window tensor:", windows.x_enc.shape)
print("deliberately hidden positions:", start, "through", start + windows.patch_length - 1)


## 4. Run pinned reconstruction and evaluate only deliberately hidden truth


In [ ]:
model = load_moment(task="reconstruction", device="cpu")
result = impute(windows, model, mask=visible, warmup=False)
metrics = masked_point_metrics(result)
provenance = build_provenance(model, windows, result)
print("masked-point MAE:", metrics.mae)
print("masked-point RMSE:", metrics.rmse)
print("scored points:", metrics.n)
print("source missing fraction:", result.masked_point_fraction)
print("model hidden fraction:", result.model_masked_point_fraction)
print("masked patch fraction:", result.masked_patch_fraction)


## 5. Visualize original versus imputed values


In [ ]:
def write_line_svg(path, layers, *, title, width=760, height=280):
    all_values = [float(value) for _, values in layers for value in values]
    low, high = min(all_values), max(all_values)
    span = high - low or 1.0
    max_points = max(len(values) for _, values in layers)
    left, right, top, bottom = 48, width - 20, 30, height - 38
    def point(index, value):
        x = left + (right - left) * index / max(max_points - 1, 1)
        y = bottom - (bottom - top) * (float(value) - low) / span
        return f"{x:.1f},{y:.1f}"
    strokes = ["#111827", "#2563eb", "#dc2626"]
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">', f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{title}</text>']
    for idx, (label, values) in enumerate(layers):
        points = " ".join(point(i, value) for i, value in enumerate(values))
        stroke = strokes[idx % len(strokes)]
        svg.append(f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points}"/>')
        svg.append(f'<text x="{left + 180 * idx}" y="{height - 10}" font-family="sans-serif" font-size="12" fill="{stroke}">{label}</text>')
    svg.append("</svg>")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(svg), encoding="utf-8")
    return path

imputed_frame = result.to_frame()
channel = windows.channels[0]
view = imputed_frame[imputed_frame["channel"] == channel].tail(96)
plot_path = write_line_svg(ROOT / "outputs" / "moment_imputation.svg", [("original", view["original_value"].fillna(view["imputed_value"]).tolist()), ("imputed", view["imputed_value"].tolist())], title=f"MOMENT imputation — {channel}")
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(plot_path)))
except ImportError:
    print(f"SVG written to {plot_path}")
print(imputed_frame.loc[imputed_frame["requested_hidden"], ["timestamp", "channel", "original_value", "imputed_value", "model_hidden"]])


## 6. Export imputed series, metrics, and provenance


In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
imputed_frame.to_csv(output_dir / "moment_imputed_series.csv", index=False)
(output_dir / "moment_imputation_metrics.json").write_text(json.dumps({"n": metrics.n, "mae": metrics.mae, "rmse": metrics.rmse}, indent=2), encoding="utf-8")
(output_dir / "moment_imputation_provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
print("exports:", sorted(path.name for path in output_dir.glob("moment_imput*")))


## Interpretation

`imputed_value` replaces only source-missing or deliberately hidden cells. Observed neighbours that MOMENT had to hide because they share the same patch are **not overwritten** in the default export. MAE/RMSE are computed only on deliberately hidden cells with genuine source truth.
